In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head()

In [ ]:
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test_data.head()

In [ ]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

In [ ]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

In [ ]:
# Cek jumlah data & tipe kolom
train_data.info()

# Statistik dasar
train_data.describe()

# Cek nilai kosong
train_data.isnull().sum()

# Cek distribusi survival
train_data['Survived'].value_counts(normalize=True)

# Cek korelasi antar fitur
train_data.corr(numeric_only=True)


In [ ]:
# Isi nilai kosong Age dengan median
train_data["Age"].fillna(train_data["Age"].median(), inplace=True)
test_data["Age"].fillna(test_data["Age"].median(), inplace=True)

# Ganti NaN di Embarked dengan 'S'
train_data["Embarked"].fillna("S", inplace=True)

# Kolom Fare juga kadang kosong di test_data
test_data["Fare"].fillna(test_data["Fare"].median(), inplace=True)

# Ubah kategori jadi angka
train_data = pd.get_dummies(train_data, columns=["Sex", "Embarked"])
test_data = pd.get_dummies(test_data, columns=["Sex", "Embarked"])


In [ ]:
features = ["Pclass", "Sex_female", "Sex_male", "Age", "SibSp", "Parch", "Fare", "Embarked_C", "Embarked_Q", "Embarked_S"]
X = train_data[features]
y = train_data["Survived"]
X_test = test_data[features]


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Pisahkan data train menjadi train/test internal
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

# Evaluasi di data validasi
y_pred = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred))


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)
grid.fit(X, y)

print("Best parameters:", grid.best_params_)


In [ ]:
final_predictions = grid.best_estimator_.predict(X_test)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': final_predictions})
output.to_csv('final_submission.csv', index=False)
print("Final submission saved!")


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Misal: kamu sudah punya model yang dilatih dengan fitur berikut
fitur = ["Pclass", "Sex_female", "Sex_male", "Age"]

# Contoh data latih (sekadar dummy)
train_data = pd.DataFrame({
    "Pclass": [1, 3, 2, 1],
    "Sex_female": [1, 0, 1, 0],
    "Sex_male": [0, 1, 0, 1],
    "Age": [22, 35, 26, 40],
    "Survived": [1, 0, 1, 0]
})

# Pisahkan fitur dan label
X = train_data[fitur]
y = train_data["Survived"]

# Latih model sederhana
model = RandomForestClassifier(random_state=42)
model.fit(X, y)

# === UJI DATA BARU ===
# Misal kita ingin uji: Pclass=2, Sex=female, Age=25
data_baru = pd.DataFrame({
    'Pclass': [2],
    'Sex_female': [1],
    'Sex_male': [0],
    'Age': [25]
})

# Prediksi
prediksi = model.predict(data_baru)
print("Prediksi:", "✅ Selamat" if prediksi[0] == 1 else "❌ Tidak selamat")
